In [1]:
import torch
import sys
sys.path.append("src")
from game.utils import Constants, Translator
from game.state import State
from game.game import Game


In [2]:
torch.save(Constants.initial_board(), "tests/fixtures/boards/init_board.pt")

In [3]:
state = State.initial_state()
torch.save([state.get_player_mask(False), state.get_player_mask(True)], "tests/fixtures/game_fixtures/player_masks.pt")
state.save("tests/fixtures/boards/init_board.json")

In [4]:
state.save("temp/test_state_saving.json")

In [5]:
State.from_file("temp/test_state_saving.json").state.shape

torch.Size([6, 4, 8])

In [6]:
from agents.mcts import Tree

In [7]:
game = Game.new_game()
tree = Tree(game, Constants.BATCH_SIZE)
tree.root.set(game.full_move_tensor(1), 1)

In [8]:
for _ in range(1000):
    end_node = tree.root.rollout()
    end_node.set(game.full_move_tensor(1), 1)
    root = end_node.backpropagate(end_node.mean_value, end_node._parent_child_index)


In [9]:
from timeit import timeit

In [ ]:
valid_moves = game.move_mask().nonzero()
flat_index = valid_moves[0][-1].item()  # Just an example valid index
shape = game.move_mask().shape

# Correct: wrap index as tensor
x1, y1, move_index = torch.unravel_index(torch.tensor(flat_index), shape)
x2, y2, flag = game.move_from_index(move_index)

# Time game.copy()
copy_time = timeit.timeit('game.copy()', globals=globals(), number=1000)
print(f"Game.copy() avg time: {copy_time / 1000:.6f} sec")

# Time game.make_move()
def make_move_test():
    g = game.copy()  # simulate what happens in your _new_node()
    g.make_move(x1, y1, x2, y2, flag)

move_time = timeit.timeit(make_move_test, number=1000)
print(f"Game.make_move() avg time (including copy): {move_time / 1000:.6f} sec")

AttributeError: 'int' object has no attribute 'is_complex'